# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
!git clone https://github.com/Asadnaeem23/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 104 (delta 24), reused 74 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 1.85 MiB | 16.87 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [18]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [19]:
!ls data/raw

content_refresh_anonymized.csv


In [20]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking**

The Content Refresh lane is best framed as a **ranking** task. The goal is to rank existing content items by how strongly the available performance and freshness signals suggest that they may need a refresh. The output would be a prioritized list for content review rather than a simple yes/no prediction.


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the main signals available for the Content Refresh task
refresh_signals = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
]

df[refresh_signals].head()


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count
0,187,20,3803,10.6,0.76,3221.0
1,445,25,15320,20.3,0.05,2481.0
2,141,20,12581,36.5,0.09,3515.0
3,463,22,11751,6.2,0.49,NaN
4,263,14,19140,44.0,0.13,2803.0


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / proxy**

There is no directly observed label in the starter data showing whether a content item actually benefited from being refreshed. For this framing, I would use a **defined refresh-priority proxy** based on observable signals such as content age, time since the last update, recent performance change, and search performance.

This is a proxy rather than an observed outcome. In a production system, the target could later be based on measured outcomes after a refresh, such as improvement in impressions, clicks, or ranking position.


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
proxy_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "avg_position",
    "ctr",
    "trend_direction",
]

df[proxy_columns].head(10)

,content_age_days,days_since_last_update,impressions_90d,impressions_last_30d,impressions_prev_30d,clicks_last_30d,clicks_prev_30d,avg_position,ctr,trend_direction
0,187,20,3803,578,987,2,13,10.6,0.76,down
1,445,25,15320,2501,5915,2,1,20.3,0.05,down
2,141,20,12581,2382,6089,1,3,36.5,0.09,down
3,463,22,11751,3626,4206,22,17,6.2,0.49,stable
4,263,14,19140,4211,6452,10,2,44.0,0.13,down
5,147,20,3970,617,1009,0,1,8.5,0.03,down
6,90,20,20,1,13,0,0,7.0,0.00,down
7,445,22,1724,636,632,1,0,21.2,0.06,stable
8,90,20,32574,5696,13828,9,8,46.0,0.09,down
9,257,104,1240,252,356,0,0,4.9,0.16,down


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Precision@K**

I would use **Precision@K** as the main ranking metric. It measures how many of the top K items selected by the ranking are relevant refresh candidates. A higher value means the prioritized list contains fewer low-value recommendations.

For example, Precision@20 would ask: out of the 20 highest-priority content items, how many are judged to be useful refresh candidates? This metric is practical because the content team has limited review capacity and needs a useful shortlist rather than a prediction for every item.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Example review sizes for the ranking
k_values = [20, 50]

print("Ranking evaluation will focus on top-K items:", k_values)


Ranking evaluation will focus on top-K items: [20, 50]


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis**

The unit of analysis is **one content item for one client**. Each row represents an individual piece of content with its search, traffic, freshness, and performance measurements. The `content_id` identifies the content item, while `client_id` identifies the client it belongs to.


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis as a real dataframe
unit_columns = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "ctr",
    "trend_direction",
]

unit_df = df[unit_columns].head(10)

print(unit_df)

print("Rows shown:", len(unit_df))
print("Unique content IDs:", unit_df["content_id"].nunique())

             content_id          client_id     content_type  content_age_days  \
0  content_304f48230142  client_f369cb89fc  keyword article               187   
1  content_a1fb4e703a9e  client_4e07408562  keyword article               445   
2  content_9aa793d4d895  client_7f2253d7e2  keyword article               141   
3  content_331d6c4de07b  client_19581e27de  keyword article               463   
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article               263   
5  content_d4084a4bc775  client_f369cb89fc  keyword article               147   
6  content_9a34b442b552  client_8722616204  keyword article                90   
7  content_a63219c6e95a  client_19581e27de  keyword article               445   
8  content_5e6c160719bc  client_6208ef0f77  keyword article                90   
9  content_c27558df2b0c  client_19581e27de  keyword article               257   

   days_since_last_update  impressions_90d  clicks_90d  avg_position   ctr  \
0                      20     

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a fixed rule**

A fixed rule such as "refresh every article older than 365 days" is useful as a simple baseline, but it does not account for the different performance and freshness patterns across content. An older article may still perform well, while a newer article may show declining impressions or poor search performance.

ML can combine several measured signals at the same time and learn patterns associated with higher refresh priority. The output can then support a ranked review list for the content team. The ML approach should be compared against simple rules rather than assuming it will always be better.


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show how refresh decisions can involve multiple signals
comparison_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "trend_direction",
]

df[comparison_columns].describe(include="all")


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction
count,30000.00000,30000.000000,30000.000000,30000.00000,30000.000000,30000
unique,NaN,NaN,NaN,NaN,NaN,5
top,NaN,NaN,NaN,NaN,NaN,down
freq,NaN,NaN,NaN,NaN,NaN,16262
mean,256.16780,46.098300,5200.366300,16.34238,0.510733,NaN
std,132.70793,42.078709,16838.019547,15.21679,3.279162,NaN
min,90.00000,1.000000,1.000000,0.00000,0.000000,NaN
25%,132.00000,20.000000,81.000000,6.20000,0.000000,NaN
50%,236.00000,20.000000,731.000000,10.80000,0.070000,NaN
75%,333.00000,104.000000,3615.250000,22.30000,0.290000,NaN


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.